In [1]:
# install transformers
!pip install transformers torch

In [2]:
# Import Libraries
from transformers import AutoModelForCausalLM, AutoTokenizer
import torch

In [3]:
model_name = "microsoft/DialoGPT-small"

tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForCausalLM.from_pretrained(model_name)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/641 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/614 [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/351M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/149 [00:00<?, ?it/s]

The tied weights mapping and config for this model specifies to tie transformer.wte.weight to lm_head.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning
GPT2LMHeadModel LOAD REPORT from: microsoft/DialoGPT-small
Key                              | Status     |  | 
---------------------------------+------------+--+-
transformer.h.{0...11}.attn.bias | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


generation_config.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

In [9]:
print("Chatbot: Hello! I am your AI assistant. Type 'exit' to stop.")

chat_history_ids = None

while True:

    user_input = input("You: ")

    if user_input.lower() in ["exit", "quit"]:
        print("Chatbot: Goodbye! Have a great day.")
        break

    # encode user input
    new_input_ids = tokenizer.encode(
        user_input + tokenizer.eos_token,
        return_tensors='pt'
    )

    # append chat history
    bot_input_ids = torch.cat(
        [chat_history_ids, new_input_ids], dim=-1
    ) if chat_history_ids is not None else new_input_ids

    # generate response (UPDATED SETTINGS)
    chat_history_ids = model.generate(
        bot_input_ids,
        max_length=1000,
        pad_token_id=tokenizer.eos_token_id,
        attention_mask=torch.ones(bot_input_ids.shape, dtype=torch.long),
        do_sample=True,
        temperature=0.7,
        top_k=50,
        top_p=0.95
    )

    # decode response
    response = tokenizer.decode(
        chat_history_ids[:, bot_input_ids.shape[-1]:][0],
        skip_special_tokens=True
    )

    print("Chatbot:", response)

Chatbot: Hello! I am your AI assistant. Type 'exit' to stop.
You: Hello
Chatbot: It's been a while since I've seen you. But hello.
You: what are you?
Chatbot: Hello. You are beautiful.
You: tell me a joke
Chatbot: What joke?
You: exit
Chatbot: Goodbye! Have a great day.


## Chatbot using Hugging Face Transformers

This chatbot uses a pre-trained DialoGPT model from Hugging Face.
The model generates responses dynamically using transformer architecture.
User input is processed and passed to the model.
The chatbot maintains conversation history for better responses.
The loop continues until the user types exit.

User Input --> Tokenization --> Transformer Model --> Response Generation --> Output --> Loop

## Observation
The chatbot generates probabilistic responses using a pretrained DialoGPT transformer model.
Responses may vary and sometimes appear unusual because the model is trained on conversational data and not fine-tuned for factual accuracy.